# 02_preprocessing.ipynb

**Project:** Predictive Maintenance - Engine Failure Prediction
**Author:** Leart Ajro
**Purpose:** This notebook will:
             - Prepare data for training/testing and modeling in future notebooks
             - Clean up unnecessary data
             - Redefine X and y variables
             - Handle class imbalance with SMOTE

## Cleaning Up Data

Since I'm only using the main sensor features such as Air temp, rotational speed, torque etc. I will drop UDI and Product Id because they are identifiers with no predictve signal (no purpose in feeding it into the model). As for the failure mode columns I will be dropping those specifically because they cause **data leakage** and extra noise.

With these included, the model would be learning from information that wouldn't exist in a real scenario. And in real production you wouldn't know which failure mode caused a breakdown until after it happened.

In [14]:
# Import packages

import pandas as pd 
import numpy as np
from imblearn.over_sampling import SMOTE

In [15]:
# Load dataset

df = pd.read_csv('../data/ai4i2020.csv')

In [16]:
# Drop unnecessary columns

df.drop(['UDI', 'Product ID', 'Type', 'TWF',
          'HDF', 'OSF', 'PWF', 'RNF'],axis=1, inplace=True)

In [17]:
# Quick view of first few rows 

df.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,298.1,308.6,1551,42.8,0,0
1,298.2,308.7,1408,46.3,3,0
2,298.1,308.5,1498,49.4,5,0
3,298.2,308.6,1433,39.5,7,0
4,298.2,308.7,1408,40.0,9,0


In [18]:
# Dimensions of dataset

df.shape

(10000, 6)

## Redefine X and Y

Now that the df has been permanently changed by dropping the unnecessary columns, I will be redefining X and Y once again.

In [19]:
# Redefine target and feature variables

target = 'Machine failure'

features = ['Air temperature [K]', 
            'Process temperature [K]',
            'Rotational speed [rpm]',
            'Torque [Nm]',
            'Tool wear [min]',
           ]
X = df[features]
y = df[target]


## Class Imbalance 

Our dataset is extremely imbalanced. with a non failure percentage of (96.61%) and failure of (3.39%), this will need to be handled before doing any modeling. The importance in handling imbalanced data is to give the model a chance to learn the minority class. With very little failures comes little information.

To fix this I will be using SMOTE (Synthetic Minority Oversampling Technique) to generate synthetic failure examples, giving the model a more balanced dataset to learn from during the training phase.

Optimizing for a metric like **recall** is also intuitive for this kind of model. In a real work environment handling machines, false negatives can become very costly.


In [20]:
# Handle class imbalance using SMOTE

smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(X,y)

In [21]:
# View value counts of resampled minority class

print(y_resampled.value_counts())

Machine failure
0    9661
1    9661
Name: count, dtype: int64


## SMOTE Test 

Running SMOTE was a success, generating snythetic failure examples until both classes are equal at 9,661 each. Creating a balance dataset giving the model a fair chance to learn both classes. 

Now we are ready to model.


In [22]:
# Save resampled data to CSV for use in notebook 3

X_resampled.to_csv('../data/X_resampled.csv', index=False)
y_resampled.to_csv('../data/y_resampled.csv', index=False)